In [1]:
#Starts from here
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression  # LogisticRegression is not used for regression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler 
from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import train_test_split
from tqdm import tqdm

def remove_low_variance_columns(df, threshold=0.005):
    # df = df.drop(['ID','SMILES','Permeability'],axis=1)
    variances = df.var()
    
    # Identify columns with variance below the threshold
    low_variance_columns = variances[variances < threshold].index.tolist()
    
    df_cleaned = df.drop(columns=low_variance_columns)
    
    return df_cleaned, low_variance_columns

def features(df, target_column='Permeability', threshold=0.9):
    correlation_matrix = df.corr()
    
    features_to_drop = set()
    
    for feature in correlation_matrix.columns:
        if feature == target_column:
            continue 
        target_corr = correlation_matrix[target_column][feature]
        
        for other_feature in correlation_matrix.columns:
            if other_feature == feature or other_feature == target_column:
                continue
            
            if abs(correlation_matrix[feature][other_feature]) > threshold:
                other_target_corr = correlation_matrix[target_column][other_feature]

                if abs(other_target_corr) < abs(target_corr):
                    features_to_drop.add(other_feature)
                else:
                    features_to_drop.add(feature)
    selected_features = [col for col in df.columns if col not in features_to_drop and col != target_column]
    
    return selected_features

In [2]:
df_desc_train = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Descriptors/Train_2d_3d_all_descriptors_Caco2.csv')
df_train = df_desc_train.sort_values(by='ID')
df_train =df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
df_desc_train = pd.concat( [df_train[['ID','SMILES','Permeability']],df_train[selected_features] ], axis=1)
df_desc_test = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Descriptors/Test_2d_3d_all_descriptors_Caco2.csv')
df_desc_test = df_desc_test.sort_values(by='ID')
df_desc_test =df_desc_test.dropna()
df_desc_test =  df_desc_test[df_desc_train.columns]


# Fingerprints
df_fp_train = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Fingerprints/Train/All_fingerprints_train_Caco2.csv')
df_train = df_fp_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
df_fp_train = pd.concat( [df_train[['ID','SMILES','Permeability']],df_train[selected_features] ], axis=1)
df_fp_test = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Fingerprints/Test/All_fingerprints_test_Caco2.csv')
df_fp_test = df_fp_test.sort_values(by='ID')
df_fp_test = df_fp_test.dropna()
df_fp_test =  df_fp_test[df_fp_train.columns]


#Smiles Embeddings
df_emb_train = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Embeddings/Train_MoLFormer-XL-both-10pct_model_1_fine_tuned_embeddings_caco2.csv')
df_train = df_emb_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
df_emb_train = pd.concat( [df_train[['ID','SMILES','Permeability']],df_train[selected_features] ], axis=1)
df_emb_test = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Embeddings/Test_MoLFormer-XL-both-10pct_model_1_fine_tuned_embeddings_caco2.csv')
df_emb_test = df_emb_test.sort_values(by='ID')
df_emb_test = df_emb_test.dropna()
df_emb_test =  df_emb_test[df_emb_train.columns]

#ATomic features
df_atomic_train = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Atomic/Train_all_atomic_desc_Caco2.csv')
df_train = df_atomic_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
df_atomic_train = pd.concat( [df_train[['ID','SMILES','Permeability']],df_train[selected_features] ], axis=1)
# df_atomic_train =pd.concat( [df_train['SMILES'], df_train.select_dtypes(include=['number'])], axis=1)
df_atomic_test = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Atomic/Test_all_atomic_desc_Caco2.csv')
df_atomic_test = df_atomic_test.sort_values(by='ID')
df_atomic_test = df_atomic_test.dropna()
df_atomic_test =  df_atomic_test[df_atomic_train.columns]


print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print('Data Loading completed')
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
df_fp_test = df_fp_test[df_fp_test['ID'].isin(df_desc_test['ID'])]
df_fp_train = df_fp_train[df_fp_train['ID'].isin(df_desc_train['ID'])]

df_emb_test = df_emb_test[df_emb_test['ID'].isin(df_desc_test['ID'])]
df_emb_train = df_emb_train[df_emb_train['ID'].isin(df_desc_train['ID'])]

df_atomic_test = df_atomic_test[df_atomic_test['ID'].isin(df_desc_test['ID'])]
df_atomic_train = df_atomic_train[df_atomic_train['ID'].isin(df_desc_train['ID'])]
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_desc_train.shape)
print(df_desc_test.shape)
print(df_fp_train.shape)
print(df_fp_test.shape)
print(df_emb_train.shape)
print(df_emb_test.shape)
print(df_atomic_train.shape)
print(df_atomic_test.shape)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')


/tmp/ipykernel_3168279/1550735325.py:1: DtypeWarning: Columns (1275,1277,1280,1285,1298,1354,1356,1359,1364,1377,1579,1580,1581,1583,1584,1596,1597) have mixed types. Specify dtype option on import or set low_memory=False.
  df_desc_train = pd.read_csv('/home/users/akshay/PCPpred/Caco2/features/Descriptors/Train_2d_3d_all_descriptors_Caco2.csv')


XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
Data Loading completed
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
(1007, 264)
(252, 264)
(1007, 868)
(252, 868)
(1007, 759)
(252, 759)
(1007, 12)
(252, 12)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX


In [3]:
merge_keys = ['ID', 'SMILES', 'Permeability']

merged_train = df_desc_train.merge(df_fp_train, on=merge_keys)
merged_train = merged_train.merge(df_emb_train, on=merge_keys)
merged_train = merged_train.merge(df_atomic_train, on=merge_keys)

merged_test = df_desc_test.merge(df_fp_test, on=merge_keys)
merged_test = merged_test.merge(df_emb_test, on=merge_keys)
merged_test = merged_test.merge(df_atomic_test, on=merge_keys)

In [4]:
X_train = merged_train.drop(columns=['ID', 'SMILES']).select_dtypes(include=['number'])
selected_final_features = features(X_train, target_column='Permeability')

train = pd.concat([merged_train[['ID', 'SMILES', 'Permeability']], X_train[selected_final_features]], axis=1)
test = merged_test[train.columns] 

print('selected_final_features', selected_final_features )
print("Final Train shape:", train.shape)
print("Final Test shape:", test.shape)

selected_final_features ['qed', 'SPS', 'FpDensityMorgan1', 'BCUT2D_MRHI', 'AvgIpc', 'BalabanJ_x', 'Ipc', 'EState_VSA11', 'NumSaturatedHeterocycles', 'NumUnspecifiedAtomStereoCenters', 'fr_Al_OH_noTert', 'fr_Ar_N', 'fr_aryl_methyl', 'fr_bicyclic', 'fr_methoxy', 'fr_morpholine', 'fr_para_hydroxylation', 'fr_piperdine', 'fr_unbrch_alkane', 'BasicGroupCount', 'AdjacencyMatrix.6', 'AdjacencyMatrix.9', 'AATS', 'AATS.4', 'AATS.23', 'AATS.90', 'AATS.95', 'AATS.96', 'ATSC.2', 'ATSC.5', 'ATSC.8', 'ATSC.11', 'ATSC.13', 'ATSC.14', 'ATSC.16', 'ATSC.17', 'ATSC.20', 'ATSC.21', 'ATSC.22', 'ATSC.23', 'ATSC.24', 'ATSC.25', 'ATSC.26', 'ATSC.28', 'ATSC.34', 'ATSC.35', 'ATSC.41', 'ATSC.42', 'ATSC.43', 'ATSC.46', 'ATSC.53', 'ATSC.64', 'ATSC.67', 'ATSC.74', 'ATSC.82', 'ATSC.84', 'ATSC.87', 'ATSC.95', 'ATSC.98', 'AATSC.9', 'AATSC.11', 'AATSC.12', 'AATSC.15', 'AATSC.16', 'AATSC.35', 'AATSC.37', 'AATSC.42', 'AATSC.47', 'AATSC.52', 'AATSC.53', 'AATSC.54', 'AATSC.58', 'AATSC.100', 'GATS.1', 'GATS.3', 'GATS.6', 'G

In [5]:
def train_and_test_predict(models, X_train, y_train, X_test, y_test):
    kf = KFold(n_splits=5, shuffle=True, random_state=101)
    results = {}
    predictions = []  

    for model in models:
        model_name = model.__class__.__name__
        predictions_train = []
        actual_y_train = []

        test_predictions_folds = []

        

        for train_index, val_index in kf.split(X_train):
            X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
            y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

            model.fit(X_train_fold, y_train_fold)

            y_pred_fold = model.predict(X_val_fold)
            y_pred_fold = np.clip(y_pred_fold, -10, -3.4)
            predictions_train.extend(y_pred_fold)
            actual_y_train.extend(y_val_fold)

            predictions_test_fold = model.predict(X_test)
            predictions_test_fold = np.clip(predictions_test_fold, -10, -3.4)
            test_predictions_folds.append(predictions_test_fold)


        mse_train = mean_squared_error(actual_y_train, predictions_train)
        mae_train = mean_absolute_error(actual_y_train, predictions_train)
        rmse_train = np.sqrt(mse_train)
        r2_train = r2_score(actual_y_train, predictions_train)
        pearson_train, _ = pearsonr(actual_y_train, predictions_train)
        spearman_train, _ = spearmanr(actual_y_train, predictions_train)


        predictions_test_mean = np.mean(test_predictions_folds, axis=0)
        predictions_test_std = np.std(test_predictions_folds, axis=0)

        mse_test = mean_squared_error(y_test, predictions_test_mean)
        mae_test = mean_absolute_error(y_test, predictions_test_mean)
        rmse_test = np.sqrt(mse_test)
        r2_test = r2_score(y_test, predictions_test_mean)
        print(r2_test)
        pearson_test, _ = pearsonr(y_test, predictions_test_mean)
        spearman_test, _ = spearmanr(y_test, predictions_test_mean)
        
        

        predictions.append({
            'Model': model_name,
            'Y Train pred': predictions_train,
            'Y Test actual': y_test,
            'Test prediction folds': test_predictions_folds,
            'Test Predictions Mean': predictions_test_mean,
            'Test Predictions Std': predictions_test_std,

        })

        results[model_name] = {
            'Train MSE (5 fold cv)': f"{mse_train:.4f}",
            'Train MAE (5 fold cv)': f"{mae_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train R2 (5 fold cv)': f"{r2_train:.4f}",
            'Train PCC (5 fold cv)': f"{pearson_train:.4f}",
            'Train SCC (5 fold cv)': f"{spearman_train:.4f}",
            'Test MSE': f"{mse_test:.4f}",
            'Test MAE': f"{mae_test:.4f}",
            'Test RMSE': f"{rmse_test:.4f}",
            'Test R2': f"{r2_test:.4f}",
            'Test Pearson Correlation': f"{pearson_test:.4f}",
            'Test Spearman Correlation': f"{spearman_test:.4f}",
        }

    results_df = pd.DataFrame(results).T
    predictions_df = pd.DataFrame(predictions)

    return results_df, predictions_df



In [6]:
X_train = train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
X_test = test[X_train.columns]
y_test = test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (1007, 1871)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 1871)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.056083 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 248675
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 1647
[LightGBM] [Info] Start training from score -6.187937
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No fu

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1493,0.2904,0.3864,0.7670,0.8758,0.8698,0.1855,0.3198,0.4307,0.7023,0.8382,0.7966
DecisionTreeRegressor,0.3054,0.4175,0.5526,0.5235,0.7616,0.7475,0.2162,0.3495,0.4650,0.6530,0.8129,0.7734
RandomForestRegressor,0.1492,0.2936,0.3862,0.7672,0.8768,0.8679,0.1856,0.3249,0.4308,0.7022,0.8384,0.7948
GradientBoostingRegressor,0.1510,0.2936,0.3886,0.7644,0.8743,0.8652,0.1911,0.3273,0.4371,0.6933,0.8330,0.7925
AdaBoostRegressor,0.1609,0.3180,0.4012,0.7489,0.8683,0.8586,0.2151,0.3643,0.4638,0.6547,0.8116,0.7624
XGBRegressor,0.1689,0.3146,0.4110,0.7364,0.8587,0.8520,0.1832,0.3164,0.4280,0.7060,0.8403,0.8024
ExtraTreesRegressor,0.1439,0.2873,0.3793,0.7755,0.8818,0.8757,0.1739,0.3149,0.4170,0.7209,0.8497,0.8136
LinearRegression,1.3241,0.8678,1.1507,-1.0661,0.4775,0.4951,0.6144,0.6193,0.7838,0.0140,0.6071,0.5999
KNeighborsRegressor,0.1901,0.3255,0.4360,0.7035,0.8409,0.8258,0.1761,0.3183,0.4197,0.7173,0.8482,0.8142
SVR,0.1458,0.2926,0.3818,0.7725,0.8796,0.8791,0.1639,0.3096,0.4049,0.7369,0.8610,0.8411


In [7]:
result_df.to_csv('/home/users/akshay/PCPpred/Caco2/results/combined_features/combined_features_caco2.csv')
prediction_df.to_csv('/home/users/akshay/PCPpred/Caco2/results/combined_features/prediction_combined_features_caco2.csv')

In [8]:
X = train.drop(columns=['ID', 'SMILES', 'Permeability'])
y = train['Permeability']

rf = RandomForestRegressor(n_estimators=100, random_state=101, n_jobs=-1)
rf.fit(X, y)

importances = rf.feature_importances_
feature_names = X.columns


In [9]:
#Top 10 features
n = 10  
top_10_indices = importances.argsort()[::-1][:n]  # indices of top n features
top_10_features = feature_names[top_10_indices].tolist() 

# Output the list
print("Top", 10, "features:\n")
print(top_10_features)

train_df = pd.concat([train[['ID', 'SMILES', 'Permeability']], X[top_10_features]], axis=1)
test_df = test[train.columns] 

Top 10 features:

['x_fine_emb_MFXL33', 'x_fine_emb_MFXL524', 'x_fine_emb_MFXL339', 'x_fine_emb_MFXL289', 'x_fine_emb_MFXL495', 'x_fine_emb_MFXL563', 'x_fine_emb_MFXL478', 'x_fine_emb_MFXL656', 'x_fine_emb_MFXL632', 'TDB9s']


In [10]:
X_train = train_df.drop(['ID','SMILES','Permeability'],axis=1)
y_train = train_df['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
X_test = test_df[X_train.columns]
y_test = test_df['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (1007, 10)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 10)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000408 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2550
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 10
[LightGBM] [Info] Start training from score -6.187937
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further sp

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

0.3775978496640814


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1672,0.3140,0.4089,0.7391,0.8600,0.8464,0.2238,0.3586,0.4730,0.6409,0.8025,0.7602
DecisionTreeRegressor,0.2983,0.4145,0.5462,0.5345,0.7734,0.7608,0.2651,0.3934,0.5148,0.5746,0.7652,0.7184
RandomForestRegressor,0.1639,0.3121,0.4048,0.7443,0.8628,0.8490,0.2201,0.3569,0.4692,0.6468,0.8046,0.7588
GradientBoostingRegressor,0.1618,0.3076,0.4023,0.7475,0.8647,0.8514,0.2179,0.3574,0.4668,0.6503,0.8072,0.7601
AdaBoostRegressor,0.1809,0.3361,0.4254,0.7177,0.8504,0.8342,0.2434,0.3944,0.4933,0.6094,0.7842,0.7270
XGBRegressor,0.1879,0.3335,0.4334,0.7068,0.8425,0.8327,0.2344,0.3661,0.4842,0.6238,0.7917,0.7489
ExtraTreesRegressor,0.1658,0.3104,0.4071,0.7414,0.8611,0.8512,0.2127,0.3504,0.4612,0.6586,0.8120,0.7662
LinearRegression,0.1697,0.3262,0.4120,0.7352,0.8574,0.8503,0.2552,0.3969,0.5052,0.5904,0.7708,0.7295
KNeighborsRegressor,0.1903,0.3341,0.4363,0.7030,0.8406,0.8299,0.2194,0.3581,0.4684,0.6479,0.8093,0.7620
SVR,0.1599,0.3039,0.3999,0.7505,0.8665,0.8558,0.2069,0.3423,0.4548,0.6680,0.8179,0.7754


In [11]:
result_df.to_csv('/home/users/akshay/PCPpred/Caco2/results/combined_features/combined_top_10_features_caco2.csv')
prediction_df.to_csv('/home/users/akshay/PCPpred/Caco2/results/combined_features/prediction_combined_top_10_features_caco2.csv')

In [12]:
#Top 20 features
n = 20  
top_20_indices = importances.argsort()[::-1][:n]  
top_20_features = feature_names[top_20_indices].tolist()  # convert to list

# Output the list
print("Top", 20, "features:\n")
print(top_20_features)

train_df = pd.concat([train[['ID', 'SMILES', 'Permeability']], X[top_20_features]], axis=1)
test_df = test[train.columns] 

Top 20 features:

['x_fine_emb_MFXL33', 'x_fine_emb_MFXL524', 'x_fine_emb_MFXL339', 'x_fine_emb_MFXL289', 'x_fine_emb_MFXL495', 'x_fine_emb_MFXL563', 'x_fine_emb_MFXL478', 'x_fine_emb_MFXL656', 'x_fine_emb_MFXL632', 'TDB9s', 'x_fine_emb_MFXL754', 'x_fine_emb_MFXL765', 'x_fine_emb_MFXL596', 'x_fine_emb_MFXL191', 'x_fine_emb_MFXL586', 'x_fine_emb_MFXL93', 'x_fine_emb_MFXL230', 'x_fine_emb_MFXL380', 'x_fine_emb_MFXL279', 'x_fine_emb_MFXL352']


In [13]:
X_train = train_df.drop(['ID','SMILES','Permeability'],axis=1)
y_train = train_df['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
X_test = test_df[X_train.columns]
y_test = test_df['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (1007, 20)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 20)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001105 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5100
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 20
[LightGBM] [Info] Start training from score -6.187937
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further sp

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

0.18162550020626778


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1538,0.2987,0.3921,0.7601,0.8719,0.8596,0.2106,0.3476,0.4589,0.6620,0.8150,0.7645
DecisionTreeRegressor,0.2953,0.4086,0.5435,0.5392,0.7680,0.7628,0.2492,0.3724,0.4992,0.6001,0.7790,0.7313
RandomForestRegressor,0.1505,0.2954,0.3880,0.7651,0.8748,0.8639,0.1992,0.3379,0.4463,0.6804,0.8249,0.7744
GradientBoostingRegressor,0.1597,0.3029,0.3996,0.7508,0.8666,0.8546,0.2032,0.3431,0.4507,0.6739,0.8215,0.7686
AdaBoostRegressor,0.1701,0.3245,0.4124,0.7346,0.8603,0.8468,0.2332,0.3780,0.4830,0.6257,0.7941,0.7407
XGBRegressor,0.1730,0.3146,0.4159,0.7301,0.8558,0.8457,0.2057,0.3425,0.4535,0.6699,0.8194,0.7708
ExtraTreesRegressor,0.1516,0.2952,0.3894,0.7634,0.8738,0.8650,0.1903,0.3290,0.4362,0.6946,0.8335,0.7864
LinearRegression,0.1571,0.3100,0.3964,0.7548,0.8688,0.8633,0.2421,0.3791,0.4920,0.6115,0.7849,0.7470
KNeighborsRegressor,0.1850,0.3303,0.4302,0.7113,0.8466,0.8324,0.1980,0.3337,0.4449,0.6823,0.8286,0.7838
SVR,0.1530,0.2955,0.3912,0.7612,0.8728,0.8641,0.1957,0.3344,0.4424,0.6859,0.8291,0.7831


In [14]:
result_df.to_csv('/home/users/akshay/PCPpred/Caco2/results/combined_features/combined_top_20_features_caco2.csv')
prediction_df.to_csv('/home/users/akshay/PCPpred/Caco2/results/combined_features/prediction_combined_top_20_features_caco2.csv')

In [15]:
#Top 50 features
n = 50  
top_50_indices = importances.argsort()[::-1][:n] 
top_50_features = feature_names[top_50_indices].tolist()  # convert to list

# Output the list
print("Top", 50, "features:\n")
print(top_50_features)

train_df = pd.concat([train[['ID', 'SMILES', 'Permeability']], X[top_50_features]], axis=1)
test_df = test[train.columns] 

Top 50 features:

['x_fine_emb_MFXL33', 'x_fine_emb_MFXL524', 'x_fine_emb_MFXL339', 'x_fine_emb_MFXL289', 'x_fine_emb_MFXL495', 'x_fine_emb_MFXL563', 'x_fine_emb_MFXL478', 'x_fine_emb_MFXL656', 'x_fine_emb_MFXL632', 'TDB9s', 'x_fine_emb_MFXL754', 'x_fine_emb_MFXL765', 'x_fine_emb_MFXL596', 'x_fine_emb_MFXL191', 'x_fine_emb_MFXL586', 'x_fine_emb_MFXL93', 'x_fine_emb_MFXL230', 'x_fine_emb_MFXL380', 'x_fine_emb_MFXL279', 'x_fine_emb_MFXL352', 'AATS.4', 'x_fine_emb_MFXL617', 'x_fine_emb_MFXL312', 'x_fine_emb_MFXL606', 'x_fine_emb_MFXL512', 'x_fine_emb_MFXL387', 'x_fine_emb_MFXL533', 'x_fine_emb_MFXL248', 'x_fine_emb_MFXL447', 'AATS.23', 'x_fine_emb_MFXL220', 'x_fine_emb_MFXL732', 'AtomTypeEState.252', 'TDB8s', 'x_fine_emb_MFXL281', 'x_fine_emb_MFXL577', 'x_fine_emb_MFXL722', 'x_fine_emb_MFXL391', 'x_fine_emb_MFXL131', 'x_fine_emb_MFXL510', 'x_fine_emb_MFXL237', 'x_fine_emb_MFXL45', 'meanI', 'x_fine_emb_MFXL204', 'x_fine_emb_MFXL85', 'x_fine_emb_MFXL292', 'x_fine_emb_MFXL593', 'x_fine_emb_M

In [16]:
X_train = train_df.drop(['ID','SMILES','Permeability'],axis=1)
y_train = train_df['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
X_test = test_df[X_train.columns]
y_test = test_df['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (1007, 50)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 50)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001954 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12750
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 50
[LightGBM] [Info] Start training from score -6.187937
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further s

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

0.25685018099685064


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1524,0.2946,0.3904,0.7622,0.8731,0.8659,0.1935,0.3331,0.4399,0.6894,0.8308,0.7859
DecisionTreeRegressor,0.3012,0.4153,0.5489,0.5300,0.7635,0.7544,0.2231,0.3497,0.4723,0.6420,0.8038,0.7572
RandomForestRegressor,0.1452,0.2921,0.3811,0.7734,0.8797,0.8707,0.1955,0.3339,0.4422,0.6862,0.8284,0.7835
GradientBoostingRegressor,0.1523,0.2930,0.3903,0.7623,0.8731,0.8631,0.1978,0.3401,0.4447,0.6826,0.8266,0.7796
AdaBoostRegressor,0.1595,0.3168,0.3993,0.7512,0.8690,0.8553,0.2200,0.3705,0.4691,0.6469,0.8058,0.7523
XGBRegressor,0.1697,0.3091,0.4120,0.7352,0.8584,0.8510,0.1985,0.3396,0.4456,0.6814,0.8258,0.7828
ExtraTreesRegressor,0.1462,0.2913,0.3823,0.7719,0.8787,0.8720,0.1857,0.3257,0.4309,0.7020,0.8379,0.7921
LinearRegression,0.1509,0.3081,0.3884,0.7646,0.8746,0.8700,0.2187,0.3607,0.4676,0.6491,0.8087,0.7755
KNeighborsRegressor,0.1696,0.3104,0.4119,0.7353,0.8593,0.8487,0.1839,0.3273,0.4288,0.7049,0.8413,0.7939
SVR,0.1469,0.2908,0.3833,0.7708,0.8781,0.8725,0.1841,0.3261,0.4290,0.7046,0.8406,0.8029


In [17]:
result_df.to_csv('/home/users/akshay/PCPpred/Caco2/results/combined_features/combined_top_50_features_caco2.csv')
prediction_df.to_csv('/home/users/akshay/PCPpred/Caco2/results/combined_features/prediction_combined_top_50_features_caco2.csv')

In [18]:
#Top 100 features
n = 100  
top_100_indices = importances.argsort()[::-1][:n]  # indices of top n features
top_100_features = feature_names[top_100_indices].tolist()  # convert to list

# Output the list
print("Top", 100, "features:\n")
print(top_100_features)

train_df = pd.concat([train[['ID', 'SMILES', 'Permeability']], X[top_100_features]], axis=1)
test_df = test[train.columns] 

Top 100 features:

['x_fine_emb_MFXL33', 'x_fine_emb_MFXL524', 'x_fine_emb_MFXL339', 'x_fine_emb_MFXL289', 'x_fine_emb_MFXL495', 'x_fine_emb_MFXL563', 'x_fine_emb_MFXL478', 'x_fine_emb_MFXL656', 'x_fine_emb_MFXL632', 'TDB9s', 'x_fine_emb_MFXL754', 'x_fine_emb_MFXL765', 'x_fine_emb_MFXL596', 'x_fine_emb_MFXL191', 'x_fine_emb_MFXL586', 'x_fine_emb_MFXL93', 'x_fine_emb_MFXL230', 'x_fine_emb_MFXL380', 'x_fine_emb_MFXL279', 'x_fine_emb_MFXL352', 'AATS.4', 'x_fine_emb_MFXL617', 'x_fine_emb_MFXL312', 'x_fine_emb_MFXL606', 'x_fine_emb_MFXL512', 'x_fine_emb_MFXL387', 'x_fine_emb_MFXL533', 'x_fine_emb_MFXL248', 'x_fine_emb_MFXL447', 'AATS.23', 'x_fine_emb_MFXL220', 'x_fine_emb_MFXL732', 'AtomTypeEState.252', 'TDB8s', 'x_fine_emb_MFXL281', 'x_fine_emb_MFXL577', 'x_fine_emb_MFXL722', 'x_fine_emb_MFXL391', 'x_fine_emb_MFXL131', 'x_fine_emb_MFXL510', 'x_fine_emb_MFXL237', 'x_fine_emb_MFXL45', 'meanI', 'x_fine_emb_MFXL204', 'x_fine_emb_MFXL85', 'x_fine_emb_MFXL292', 'x_fine_emb_MFXL593', 'x_fine_emb_

In [19]:
X_train = train_df.drop(['ID','SMILES','Permeability'],axis=1)
y_train = train_df['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
X_test = test_df[X_train.columns]
y_test = test_df['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (1007, 100)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 100)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003419 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 25330
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 100
[LightGBM] [Info] Start training from score -6.187937
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furthe

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

0.2214725837399104


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1473,0.2892,0.3838,0.7702,0.8776,0.8684,0.1900,0.3253,0.4359,0.6950,0.8339,0.7942
DecisionTreeRegressor,0.2779,0.3997,0.5271,0.5664,0.7769,0.7622,0.2216,0.3474,0.4707,0.6444,0.8050,0.7714
RandomForestRegressor,0.1419,0.2883,0.3767,0.7785,0.8827,0.8722,0.1910,0.3278,0.4370,0.6935,0.8328,0.7889
GradientBoostingRegressor,0.1452,0.2886,0.3810,0.7735,0.8795,0.8688,0.1882,0.3289,0.4339,0.6979,0.8356,0.7946
AdaBoostRegressor,0.1541,0.3116,0.3926,0.7595,0.8740,0.8621,0.2190,0.3682,0.4680,0.6485,0.8070,0.7575
XGBRegressor,0.1623,0.3088,0.4029,0.7467,0.8645,0.8545,0.1925,0.3308,0.4387,0.6911,0.8314,0.7946
ExtraTreesRegressor,0.1426,0.2864,0.3776,0.7775,0.8821,0.8761,0.1841,0.3230,0.4291,0.7045,0.8395,0.7967
LinearRegression,0.1527,0.3088,0.3908,0.7617,0.8736,0.8707,0.2132,0.3511,0.4617,0.6579,0.8146,0.7873
KNeighborsRegressor,0.1732,0.3120,0.4162,0.7297,0.8567,0.8479,0.1821,0.3148,0.4268,0.7077,0.8436,0.8055
SVR,0.1425,0.2877,0.3775,0.7777,0.8819,0.8779,0.1786,0.3167,0.4226,0.7134,0.8454,0.8125


In [20]:
result_df.to_csv('/home/users/akshay/PCPpred/Caco2/results/combined_features/combined_top_100_features_caco2.csv')
prediction_df.to_csv('/home/users/akshay/PCPpred/Caco2/results/combined_features/prediction_combined_top_100_features_caco2.csv')

In [21]:
#Top 200 features
n = 200  
top_200_indices = importances.argsort()[::-1][:n]  # indices of top n features
top_200_features = feature_names[top_200_indices].tolist()  # convert to list

# Output the list
print("Top", 200, "features:\n")
print(top_200_features)

train_df = pd.concat([train[['ID', 'SMILES', 'Permeability']], X[top_200_features]], axis=1)
test_df = test[train.columns]

Top 200 features:

['x_fine_emb_MFXL33', 'x_fine_emb_MFXL524', 'x_fine_emb_MFXL339', 'x_fine_emb_MFXL289', 'x_fine_emb_MFXL495', 'x_fine_emb_MFXL563', 'x_fine_emb_MFXL478', 'x_fine_emb_MFXL656', 'x_fine_emb_MFXL632', 'TDB9s', 'x_fine_emb_MFXL754', 'x_fine_emb_MFXL765', 'x_fine_emb_MFXL596', 'x_fine_emb_MFXL191', 'x_fine_emb_MFXL586', 'x_fine_emb_MFXL93', 'x_fine_emb_MFXL230', 'x_fine_emb_MFXL380', 'x_fine_emb_MFXL279', 'x_fine_emb_MFXL352', 'AATS.4', 'x_fine_emb_MFXL617', 'x_fine_emb_MFXL312', 'x_fine_emb_MFXL606', 'x_fine_emb_MFXL512', 'x_fine_emb_MFXL387', 'x_fine_emb_MFXL533', 'x_fine_emb_MFXL248', 'x_fine_emb_MFXL447', 'AATS.23', 'x_fine_emb_MFXL220', 'x_fine_emb_MFXL732', 'AtomTypeEState.252', 'TDB8s', 'x_fine_emb_MFXL281', 'x_fine_emb_MFXL577', 'x_fine_emb_MFXL722', 'x_fine_emb_MFXL391', 'x_fine_emb_MFXL131', 'x_fine_emb_MFXL510', 'x_fine_emb_MFXL237', 'x_fine_emb_MFXL45', 'meanI', 'x_fine_emb_MFXL204', 'x_fine_emb_MFXL85', 'x_fine_emb_MFXL292', 'x_fine_emb_MFXL593', 'x_fine_emb_

In [22]:
X_train = train_df.drop(['ID','SMILES','Permeability'],axis=1)
y_train = train_df['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
X_test = test_df[X_train.columns]
y_test = test_df['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (1007, 200)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 200)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006840 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 50830
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 200
[LightGBM] [Info] Start training from score -6.187937
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furthe

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


0.20940397623399531


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1451,0.2876,0.3809,0.7736,0.8795,0.8718,0.1886,0.3217,0.4343,0.6973,0.8354,0.7928
DecisionTreeRegressor,0.2877,0.3929,0.5364,0.5511,0.7723,0.7614,0.2337,0.3685,0.4834,0.6249,0.7941,0.7607
RandomForestRegressor,0.1423,0.2869,0.3772,0.7780,0.8826,0.8731,0.1893,0.3273,0.4351,0.6962,0.8345,0.7911
GradientBoostingRegressor,0.1464,0.2883,0.3827,0.7715,0.8784,0.8682,0.1902,0.3253,0.4361,0.6948,0.8340,0.7905
AdaBoostRegressor,0.1581,0.3136,0.3977,0.7533,0.8704,0.8601,0.2189,0.3653,0.4679,0.6486,0.8067,0.7583
XGBRegressor,0.1645,0.3085,0.4056,0.7433,0.8627,0.8542,0.1970,0.3326,0.4439,0.6838,0.8270,0.7886
ExtraTreesRegressor,0.1417,0.2859,0.3765,0.7788,0.8832,0.8762,0.1822,0.3186,0.4269,0.7076,0.8414,0.8007
LinearRegression,0.1682,0.3232,0.4101,0.7376,0.8614,0.8590,0.2240,0.3680,0.4733,0.6405,0.8077,0.7803
KNeighborsRegressor,0.1815,0.3239,0.4260,0.7168,0.8487,0.8372,0.1943,0.3216,0.4408,0.6881,0.8324,0.7910
SVR,0.1419,0.2863,0.3766,0.7787,0.8825,0.8815,0.1734,0.3127,0.4164,0.7217,0.8504,0.8194


In [23]:
result_df.to_csv('/home/users/akshay/PCPpred/Caco2/results/combined_features/combined_top_200_features_caco2.csv')
prediction_df.to_csv('/home/users/akshay/PCPpred/Caco2/results/combined_features/prediction_combined_top_200_features_caco2.csv')

In [24]:
#Top 500 features
n = 500  
top_500_indices = importances.argsort()[::-1][:n]  # indices of top n features
top_500_features = feature_names[top_500_indices].tolist()  # convert to list

# Output the list
print("Top", 500, "features:\n")
print(top_500_features)

train_df = pd.concat([train[['ID', 'SMILES', 'Permeability']], X[top_500_features]], axis=1)
test_df = test[train.columns]

Top 500 features:

['x_fine_emb_MFXL33', 'x_fine_emb_MFXL524', 'x_fine_emb_MFXL339', 'x_fine_emb_MFXL289', 'x_fine_emb_MFXL495', 'x_fine_emb_MFXL563', 'x_fine_emb_MFXL478', 'x_fine_emb_MFXL656', 'x_fine_emb_MFXL632', 'TDB9s', 'x_fine_emb_MFXL754', 'x_fine_emb_MFXL765', 'x_fine_emb_MFXL596', 'x_fine_emb_MFXL191', 'x_fine_emb_MFXL586', 'x_fine_emb_MFXL93', 'x_fine_emb_MFXL230', 'x_fine_emb_MFXL380', 'x_fine_emb_MFXL279', 'x_fine_emb_MFXL352', 'AATS.4', 'x_fine_emb_MFXL617', 'x_fine_emb_MFXL312', 'x_fine_emb_MFXL606', 'x_fine_emb_MFXL512', 'x_fine_emb_MFXL387', 'x_fine_emb_MFXL533', 'x_fine_emb_MFXL248', 'x_fine_emb_MFXL447', 'AATS.23', 'x_fine_emb_MFXL220', 'x_fine_emb_MFXL732', 'AtomTypeEState.252', 'TDB8s', 'x_fine_emb_MFXL281', 'x_fine_emb_MFXL577', 'x_fine_emb_MFXL722', 'x_fine_emb_MFXL391', 'x_fine_emb_MFXL131', 'x_fine_emb_MFXL510', 'x_fine_emb_MFXL237', 'x_fine_emb_MFXL45', 'meanI', 'x_fine_emb_MFXL204', 'x_fine_emb_MFXL85', 'x_fine_emb_MFXL292', 'x_fine_emb_MFXL593', 'x_fine_emb_

In [25]:
X_train = train_df.drop(['ID','SMILES','Permeability'],axis=1)
y_train = train_df['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
X_test = test_df[X_train.columns]
y_test = test_df['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (1007, 500)
y_train shape:  (1007,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (252, 500)
y_test shape:  (252,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.010584 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 127316
[LightGBM] [Info] Number of data points in the train set: 805, number of used features: 500
[LightGBM] [Info] Start training from score -6.187937
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furth

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1476,0.2908,0.3842,0.7697,0.8774,0.8692,0.1808,0.3174,0.4252,0.7098,0.8427,0.8022
DecisionTreeRegressor,0.3135,0.4223,0.5599,0.5108,0.7525,0.7379,0.2071,0.3504,0.4551,0.6676,0.8194,0.7812
RandomForestRegressor,0.1454,0.2901,0.3813,0.7731,0.8801,0.8713,0.1887,0.3285,0.4344,0.6972,0.8352,0.7928
GradientBoostingRegressor,0.1470,0.2896,0.3834,0.7706,0.8778,0.8697,0.1906,0.3254,0.4366,0.6941,0.8335,0.7924
AdaBoostRegressor,0.1601,0.3158,0.4001,0.7502,0.8688,0.8558,0.2181,0.3660,0.4671,0.6499,0.8083,0.7604
XGBRegressor,0.1718,0.3145,0.4145,0.7319,0.8560,0.8486,0.1882,0.3252,0.4338,0.6979,0.8354,0.8007
ExtraTreesRegressor,0.1453,0.2895,0.3812,0.7733,0.8803,0.8736,0.1780,0.3145,0.4219,0.7143,0.8455,0.8037
LinearRegression,0.4130,0.4749,0.6426,0.3556,0.7127,0.7226,0.3306,0.4563,0.5750,0.4694,0.7373,0.6901
KNeighborsRegressor,0.1906,0.3279,0.4366,0.7026,0.8409,0.8290,0.1867,0.3182,0.4321,0.7003,0.8386,0.8003
SVR,0.1451,0.2922,0.3809,0.7736,0.8797,0.8792,0.1741,0.3169,0.4173,0.7206,0.8500,0.8195


In [26]:
result_df.to_csv('/home/users/akshay/PCPpred/Caco2/results/combined_features/combined_top_500_features_caco2.csv')
prediction_df.to_csv('/home/users/akshay/PCPpred/Caco2/results/combined_features/prediction_combined_top_500_features_caco2.csv')